In [1]:
#%% Import packages
import numpy as np
import pandas as pd
#import random
#from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, make_scorer
from sklearn.impute import KNNImputer
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.utils import resample
# from sklearn.model_selection import StratifiedKFold

In [2]:
#%% Import data using pandas
#upload the  path of the excel file titled "Normalized_Values_Features_Solvent" 
#path = 

In [3]:
#%% Define needed fucntions
# Function to import data from given excel
def import_data(url):
  DataforML = pd.read_excel(url)
  DataforML.set_index("Chemical_Name", inplace=True)
  return DataforML

def scale_data(data):
    pass

# Function for replacing missing data 
def replace_missing_data(data, col=['Climate Change (kg CO2-eq)']):
  imputer = KNNImputer(n_neighbors = 5)
  data_re = data.copy().values.reshape(-1,1)
  data_ = imputer.fit_transform(data_re)
  data_ = pd.DataFrame(data_, columns=col, index=data.index)
  return data_

# Function to split scaled data into training, cross-validation (or development) and testing sets
def data_split(X, y, ts=0.3):
  X_train ,X_splt ,y_train ,y_splt = train_test_split(X,y,test_size=ts,random_state=1)
  X_cv ,X_test ,y_cv ,y_test = train_test_split(X_splt,y_splt,test_size=0.5,random_state=1)
  return [X_train, X_cv, X_test, y_train, y_cv, y_test]

# Function for R-squared and root mean squared error
def get_r2score_and_rmse(y_true_train, y_pred_train ,y_true_val, y_pred_val, y_true_test, y_pred_test):
    """estimate R-squared and root-mean-squared-error"""
    R2_score_train = r2_score(y_true_train, y_pred_train)
    R2_score_valid = r2_score(y_true_val, y_pred_val)
    R2_score_test  = r2_score(y_true_test, y_pred_test)
    
    RMSE_train = np.sqrt(mean_squared_error(y_true_train, y_pred_train))
    RMSE_valid = np.sqrt(mean_squared_error(y_true_val, y_pred_val))
    RMSE_test  = np.sqrt(mean_squared_error(y_true_test, y_pred_test))
    
    print("-----------------------------------------------------------------------")
    print('R-squared: train=',R2_score_train, ', validation=',R2_score_valid, ', test=',R2_score_test)
    print("-----------------------------------------------------------------------")
    print('RMSE: train=',RMSE_train, ', validation=',RMSE_valid, ', test=',RMSE_test)
    print("-----------------------------------------------------------------------")
    
    return [R2_score_train,R2_score_valid,R2_score_test,RMSE_train,RMSE_valid,RMSE_test]

def plot_results(y_true_train_HH,y_pred_train_HH,y_true_val_HH,y_pred_val_HH,y_true_test_HH,y_pred_test_HH,rmse_tr_HH,rmse_va_HH,rmse_te_HH,r2s_tr_HH,r2s_va_HH,r2s_te_HH,sel_data_HH,
                 y_true_train_EQ,y_pred_train_EQ,y_true_val_EQ,y_pred_val_EQ,y_true_test_EQ,y_pred_test_EQ,rmse_tr_EQ,rmse_va_EQ,rmse_te_EQ,r2s_tr_EQ,r2s_va_EQ,r2s_te_EQ,sel_data_EQ,    
                 y_true_train_CC,y_pred_train_CC,y_true_val_CC,y_pred_val_CC,y_true_test_CC,y_pred_test_CC,rmse_tr_CC,rmse_va_CC,rmse_te_CC,r2s_tr_CC,r2s_va_CC,r2s_te_CC,sel_data_CC,    
                 y_true_train_RU,y_pred_train_RU,y_true_val_RU,y_pred_val_RU,y_true_test_RU,y_pred_test_RU,rmse_tr_RU,rmse_va_RU,rmse_te_RU,r2s_tr_RU,r2s_va_RU,r2s_te_RU,sel_data_RU,    
                 ):
    plt.figure(figsize=(14,8), dpi=800)
    plt.subplots_adjust(hspace=0.2)
    plt.subplot(2,2,1)
    #plt.xscale("log")
    #plt.yscale("log")
    plt.plot(y_true_train_HH, y_pred_train_HH, 'b.', label=f'Training: RMSE={rmse_tr_HH:.2E}, R-squared={r2s_tr_HH:.3f}', markersize=18)
    plt.plot(y_true_val_HH, y_pred_val_HH, 'y.', label=f'Validation: RMSE={rmse_va_HH:.2E}, R-squared={r2s_va_HH:.3f}', markersize=18)
    plt.plot(y_true_test_HH, y_pred_test_HH, 'r.', label=f'Testing: RMSE={rmse_te_HH:.2E}, R-squared={r2s_te_HH:.3f}', markersize=18)
    plt.plot([sel_data_HH.iloc[:,-1].min(), sel_data_HH.iloc[:,-1].max()], [sel_data_HH.iloc[:,-1].min(), sel_data_HH.iloc[:,-1].max()], 'k--')
    plt.xlabel('Actual, (x10$^-$$^6$) DALY/$kg_c$$_h$$_e$$_m$', fontsize=13)
    plt.ylabel('Predicted, (x10$^-$$^6$) DALY/$kg_c$$_h$$_e$$_m$', fontsize=13)
    plt.legend(loc='best')
    plt.text(sel_data_HH.iloc[:,-1].max()-1, 0, '(a)', verticalalignment='bottom', horizontalalignment='left', fontsize=15)
    #plt.title('Human Health Impact')
    
    plt.subplot(2,2,2)
    #plt.xscale("log")
    #plt.yscale("log")
    plt.plot(y_true_train_EQ, y_pred_train_EQ, 'b.', label=f'Training: RMSE={rmse_tr_EQ:.2E}, R-squared={r2s_tr_EQ:.3f}', markersize=18)
    plt.plot(y_true_val_EQ, y_pred_val_EQ, 'y.', label=f'Validation: RMSE={rmse_va_EQ:.2E}, R-squared={r2s_va_EQ:.3f}', markersize=18)
    plt.plot(y_true_test_EQ, y_pred_test_EQ, 'r.', label=f'Testing: RMSE={rmse_te_EQ:.2E}, R-squared={r2s_te_EQ:.3f}', markersize=18)
    plt.plot([sel_data_EQ.iloc[:,-1].min(), sel_data_EQ.iloc[:,-1].max()], [sel_data_EQ.iloc[:,-1].min(), sel_data_EQ.iloc[:,-1].max()], 'k--')
    plt.xlabel('Actual, PDF*$m^2$*yr/$kg_c$$_h$$_e$$_m$', fontsize=13)
    plt.ylabel('Predicted, PDF*$m^2$*yr/$kg_c$$_h$$_e$$_m$', fontsize=13)
    plt.legend(loc='best')
    plt.text(sel_data_EQ.iloc[:,-1].max()-0.5, 0, '(b)', verticalalignment='bottom', horizontalalignment='left', fontsize=15)
    #plt.title('Ecosystem Quality')
    
    plt.subplot(2,2,3)
    #plt.xscale("log")
    #plt.yscale("log")
    plt.plot(y_true_train_CC, y_pred_train_CC, 'b.', label=f'Training: RMSE={rmse_tr_CC:.2E}, R-squared={r2s_tr_CC:.3f}', markersize=18)
    plt.plot(y_true_val_CC, y_pred_val_CC, 'y.', label=f'Validation: RMSE={rmse_va_CC:.2E}, R-squared={r2s_va_CC:.3f}', markersize=18)
    plt.plot(y_true_test_CC, y_pred_test_CC, 'r.', label=f'Testing: RMSE={rmse_te_CC:.2E}, R-squared={r2s_te_CC:.3f}', markersize=18)
    plt.plot([sel_data_CC.iloc[:,-1].min(), sel_data_CC.iloc[:,-1].max()], [sel_data_CC.iloc[:,-1].min(), sel_data_CC.iloc[:,-1].max()], 'k--')
    plt.xlabel('Actual, kg$CO_2$-eq/$kg_c$$_h$$_e$$_m$', fontsize=13)
    plt.ylabel('Predicted, kg$CO_2$-eq/$kg_c$$_h$$_e$$_m$', fontsize=13)
    plt.legend(loc='best')
    plt.text(sel_data_CC.iloc[:,-1].max()-1, 0, '(c)', verticalalignment='bottom', horizontalalignment='left', fontsize=15)
    #plt.title('Global Warming Potential')
    
    plt.subplot(2,2,4)
    #plt.xscale("log")
    #plt.yscale("log")
    plt.plot(y_true_train_RU, y_pred_train_RU, 'b.', label=f'Training: RMSE={rmse_tr_RU:.2E}, R-squared={r2s_tr_RU:.3f}', markersize=18)
    plt.plot(y_true_val_RU, y_pred_val_RU, 'y.', label=f'Validation: RMSE={rmse_va_RU:.2E}, R-squared={r2s_va_RU:.3f}', markersize=18)
    plt.plot(y_true_test_RU, y_pred_test_RU, 'r.', label=f'Testing: RMSE={rmse_te_RU:.2E}, R-squared={r2s_te_RU:.3f}', markersize=18)
    plt.plot([sel_data_RU.iloc[:,-1].min(), sel_data_RU.iloc[:,-1].max()], [sel_data_RU.iloc[:,-1].min(), sel_data_RU.iloc[:,-1].max()], 'k--')
    plt.xlabel('Actual, (x10$^1$) MJ/$kg_c$$_h$$_e$$_m$', fontsize=13)
    plt.ylabel('Predicted, (x10$^1$) MJ/$kg_c$$_h$$_e$$_m$', fontsize=13)
    plt.legend(loc='best')
    plt.text(sel_data_RU.iloc[:,-1].max()-1, 0, '(d)', verticalalignment='bottom', horizontalalignment='left', fontsize=15)
    #plt.title('Resource utilization')
    plt.show()

In [4]:
#%% Import data 
Data   = import_data(path)

In [5]:
#%% Select needed thermo and mol-desc features from the dataset
thermo_feat_HH   = ['Heat of Vaporization(J/mol)', 'Heat Capacity (kJ/kgC)', 'XLogP','Pitzer’s Acentric Factor [-]', 'Critical Temperature [K]']
mol_desc_feat_HH = ['Chi0n', 'HallKierAlpha', 'SMR_VSA7', 'VSA_EState6','NumValenceElectrons']
tX_data_HH = Data[thermo_feat_HH]
mX_data_HH = Data[mol_desc_feat_HH]

thermo_feat_EQ = ['Heat Capacity (kJ/kgC)', 'Boiling Point(K)','wwhatsa (Gas) [J/mol]', 'Critical Temperature [K]','Critical Molar Volume [m3/mol]']
mol_desc_feat_EQ = ['BertzCT', 'Chi2v', 'HallKierAlpha', 'qed', 'fr_halogen']
tX_data_EQ = Data[thermo_feat_EQ]
mX_data_EQ = Data[mol_desc_feat_EQ]

thermo_feat_CC   = ['Heat Capacity (kJ/kgC)', 'Boiling Point(K)', 'XLogP', 'Critical Temperature [K]', 'Critical Molar Volume [m3/mol]']
mol_desc_feat_CC = ['BertzCT', 'ExactMolWt', 'HallKierAlpha', 'PEOE_VSA6', 'NOCount']
tX_data_CC = Data[thermo_feat_CC]
mX_data_CC = Data[mol_desc_feat_CC]

thermo_feat_RU = ['Heat Capacity (kJ/kgC)', 'Boiling Point(K)', 'XLogP','Critical Pressure [kPa]', 'Critical Temperature [K]']
mol_desc_feat_RU = ['ExactMolWt', 'MaxAbsPartialCharge', 'MaxPartialCharge', 'SMR_VSA2','NumRotatableBonds']
tX_data_RU = Data[thermo_feat_RU]
mX_data_RU = Data[mol_desc_feat_RU]

KeyError: "['wwhatsa (Gas) [J/mol]'] not in index"

In [ ]:
#%% Select the "climate change" label from the dataset and replace missing data
y_data_HH  = Data["Human Health (DALY)"]
y_data_EQ  = Data["Ecosystem Quality (PDF*m2*yr)"]
y_data_CC  = Data["Climate Change (kg CO2-eq)"]
y_data_RU  = Data["Resources (MJ primary)"]
y_data_HH  = replace_missing_data(y_data_HH, col=['Human Health (DALY)'])
y_data_EQ  = replace_missing_data(y_data_EQ, col=['Ecosystem Quality (PDF*m2*yr)'])
y_data_CC  = replace_missing_data(y_data_CC, col=['Climate Change (kg CO2-eq)'])
y_data_RU  = replace_missing_data(y_data_RU, col=['Resources (MJ primary)'])

In [ ]:
#%% Combine the selected features with the label
sel_data_HH = pd.concat([tX_data_HH, mX_data_HH], axis=1)
sel_data_HH = pd.concat([sel_data_HH, y_data_HH*1000000], axis=1)

sel_data_EQ = pd.concat([tX_data_EQ, mX_data_EQ], axis=1)
sel_data_EQ = pd.concat([sel_data_EQ, y_data_EQ], axis=1)

sel_data_CC = pd.concat([tX_data_CC, mX_data_CC], axis=1)
sel_data_CC = pd.concat([sel_data_CC, y_data_CC], axis=1)

sel_data_RU = pd.concat([tX_data_RU, mX_data_RU], axis=1)
sel_data_RU = pd.concat([sel_data_RU, y_data_RU*0.1], axis=1)

In [ ]:
#%% Drop outliers
sel_data_CC = sel_data_CC.drop(sel_data_CC[sel_data_CC['Climate Change (kg CO2-eq)'] > 20].index)


In [ ]:
#%% Split the data
Xtrain_HH, Xcv_HH, Xtest_HH, ytrain_HH, ycv_HH, ytest_HH = data_split(sel_data_HH.iloc[:,:-1], 
                                                                      sel_data_HH.iloc[:,-1],
                                                                      ts=0.20)

Xtrain_EQ, Xcv_EQ, Xtest_EQ, ytrain_EQ, ycv_EQ, ytest_EQ = data_split(sel_data_EQ.iloc[:,:-1], 
                                                                      sel_data_EQ.iloc[:,-1],
                                                                      ts=0.20)

Xtrain_CC, Xcv_CC, Xtest_CC, ytrain_CC, ycv_CC, ytest_CC = data_split(sel_data_CC.iloc[:,:-1], 
                                                                      sel_data_CC.iloc[:,-1],
                                                                      ts=0.20)

Xtrain_RU, Xcv_RU, Xtest_RU, ytrain_RU, ycv_RU, ytest_RU = data_split(sel_data_RU.iloc[:,:-1], 
                                                                      sel_data_RU.iloc[:,-1],
                                                                      ts=0.25)

In [ ]:
#%% load developed models
#Open the ANN folder i nthe githb repository, go to the weight  folder inside the ANN foderad upload the path for each indicator 
#model_hh = 
#model_eq = 
#model_cc = 
#model_ru = 

In [ ]:
#%% Make predictions
Y1P_HH = model_hh.predict(Xtrain_HH, verbose=0)
Y2P_HH = model_hh.predict(Xcv_HH, verbose=0)
Y3P_HH = model_hh.predict(Xtest_HH, verbose=0)

Y1P_EQ = model_eq.predict(Xtrain_EQ, verbose=0)
Y2P_EQ = model_eq.predict(Xcv_EQ, verbose=0)
Y3P_EQ = model_eq.predict(Xtest_EQ, verbose=0)

Y1P_CC = model_cc.predict(Xtrain_CC, verbose=0)
Y2P_CC = model_cc.predict(Xcv_CC, verbose=0)
Y3P_CC = model_cc.predict(Xtest_CC, verbose=0)

Y1P_RU = model_ru.predict(Xtrain_RU, verbose=0)
Y2P_RU = model_ru.predict(Xcv_RU, verbose=0)
Y3P_RU = model_ru.predict(Xtest_RU, verbose=0)


In [ ]:
#%% Get r2_score and rmse
scoress_HH = get_r2score_and_rmse(ytrain_HH,Y1P_HH,ycv_HH,Y2P_HH,ytest_HH,Y3P_HH)
scoress_EQ = get_r2score_and_rmse(ytrain_EQ,Y1P_EQ,ycv_EQ,Y2P_EQ,ytest_EQ,Y3P_EQ)
scoress_CC = get_r2score_and_rmse(ytrain_CC,Y1P_CC,ycv_CC,Y2P_CC,ytest_CC,Y3P_CC)
scoress_RU = get_r2score_and_rmse(ytrain_RU,Y1P_RU,ycv_RU,Y2P_RU,ytest_RU,Y3P_RU) 


In [ ]:
#%% Plot results
plot_results(ytrain_HH,Y1P_HH,ycv_HH,Y2P_HH,ytest_HH,Y3P_HH,scoress_HH[3],scoress_HH[4],scoress_HH[5],scoress_HH[0],scoress_HH[1],scoress_HH[2],sel_data_HH,
             ytrain_EQ,Y1P_EQ,ycv_EQ,Y2P_EQ,ytest_EQ,Y3P_EQ,scoress_EQ[3],scoress_EQ[4],scoress_EQ[5],scoress_EQ[0],scoress_EQ[1],scoress_EQ[2],sel_data_EQ,
             ytrain_CC,Y1P_CC,ycv_CC,Y2P_CC,ytest_CC,Y3P_CC,scoress_CC[3],scoress_CC[4],scoress_CC[5],scoress_CC[0],scoress_CC[1],scoress_CC[2],sel_data_CC,
             ytrain_RU,Y1P_RU,ycv_RU,Y2P_RU,ytest_RU,Y3P_RU,scoress_RU[3],scoress_RU[4],scoress_RU[5],scoress_RU[0],scoress_RU[1],scoress_RU[2],sel_data_RU,
             )


In [ ]:
#%% Feature importance
scorer = make_scorer(mean_squared_error, greater_is_better=False)
perm_HH = permutation_importance(model_hh, Xtest_HH, ytest_HH, n_repeats=5, random_state=42, scoring=scorer)
importances_HH = perm_HH.importances_mean
idxx_HH = np.argsort(importances_HH)

perm_EQ = permutation_importance(model_eq, Xtest_EQ, ytest_EQ, n_repeats=5, random_state=42, scoring=scorer)
importances_EQ = perm_EQ.importances_mean
idxx_EQ = np.argsort(importances_EQ)

perm_CC = permutation_importance(model_cc, Xtest_CC, ytest_CC, n_repeats=5, random_state=42, scoring=scorer)
importances_CC = perm_CC.importances_mean
idxx_CC = np.argsort(importances_CC)

perm_RU = permutation_importance(model_ru, Xtest_RU, ytest_RU, n_repeats=5, random_state=42, scoring=scorer)
importances_RU = perm_RU.importances_mean
idxx_RU = np.argsort(importances_RU)


In [ ]:
#%% plot importance
plt.figure(figsize=(12,10),dpi=800)
plt.subplot(4,1,1)
plt.barh(range(len(idxx_HH)),importances_HH[idxx_HH])
#plt.xlabel("Importance")
plt.yticks(range(len(idxx_HH)), [sel_data_HH.iloc[:,:-1].columns[i] for i in idxx_HH])
plt.text(23.5, 0, '(a)', verticalalignment='bottom', horizontalalignment='left', fontsize=15)

plt.subplot(4,1,2)
plt.barh(range(len(idxx_EQ)),importances_EQ[idxx_EQ])
#plt.xlabel("Importance")
plt.yticks(range(len(idxx_EQ)), [sel_data_EQ.iloc[:,:-1].columns[i] for i in idxx_EQ])
plt.text(0.45, 0, '(b)', verticalalignment='bottom', horizontalalignment='left', fontsize=15)

plt.subplot(4,1,3)
plt.barh(range(len(idxx_CC)),importances_CC[idxx_CC])
#plt.xlabel("Importance")
plt.yticks(range(len(idxx_CC)), [sel_data_CC.iloc[:,:-1].columns[i] for i in idxx_CC])
plt.text(1.8, 0, '(c)', verticalalignment='bottom', horizontalalignment='left', fontsize=15)

plt.subplot(4,1,4)
plt.barh(range(len(idxx_RU)),importances_RU[idxx_RU])
plt.xlabel("Importance")
plt.yticks(range(len(idxx_RU)), [sel_data_RU.iloc[:,:-1].columns[i] for i in idxx_RU])
plt.text(22, 0, '(d)', verticalalignment='bottom', horizontalalignment='left', fontsize=15)
plt.show()


In [ ]:

#%% confidence interval
n_iterations = 1000  # Number of bootstrap samples to create
n_size = int(len(sel_data_HH.iloc[:,:-1]) * 0.50)  # Randomly selecting 50% of the dataset in each iteration

predictions_hh = []
for i in range(n_iterations):
    X_resample, y_resample = resample(sel_data_HH.iloc[:,:-1], sel_data_HH.iloc[:,-1], n_samples=n_size)
    y_pred_hh = model_hh.predict(X_resample)
    predictions_hh.append(y_pred_hh)
predictions_hh = np.array(predictions_hh)

predictions_eq = []
for i in range(n_iterations):
    X_resample, y_resample = resample(sel_data_EQ.iloc[:,:-1], sel_data_EQ.iloc[:,-1], n_samples=n_size)
    y_pred_eq = model_eq.predict(X_resample)
    predictions_eq.append(y_pred_eq)
predictions_eq = np.array(predictions_eq)

predictions_cc = []
for i in range(n_iterations):
    X_resample, y_resample = resample(sel_data_CC.iloc[:,:-1], sel_data_CC.iloc[:,-1], n_samples=n_size)
    y_pred_cc = model_cc.predict(X_resample)
    predictions_cc.append(y_pred_cc)
predictions_cc = np.array(predictions_cc)

predictions_ru = []
for i in range(n_iterations):
    X_resample, y_resample = resample(sel_data_RU.iloc[:,:-1], sel_data_RU.iloc[:,-1], n_samples=n_size)
    y_pred_ru = model_ru.predict(X_resample)
    predictions_ru.append(y_pred_ru)
predictions_ru = np.array(predictions_ru)

In [ ]:
#%% Compute the 95% confidence interval
lower_hh = np.percentile(predictions_hh, 2.5)
upper_hh = np.percentile(predictions_hh, 97.5)
lower_eq = np.percentile(predictions_eq, 2.5)
upper_eq = np.percentile(predictions_eq, 97.5)
lower_cc = np.percentile(predictions_cc, 2.5)
upper_cc = np.percentile(predictions_cc, 97.5)
lower_ru = np.percentile(predictions_ru, 2.5)
upper_ru = np.percentile(predictions_ru, 97.5)

print(f"95% Confidence Interval (HH): ({lower_hh:.3f}, {upper_hh:.3f})")
print(f"95% Confidence Interval (EQ): ({lower_eq:.3f}, {upper_eq:.3f})")
print(f"95% Confidence Interval (CC): ({lower_cc:.3f}, {upper_cc:.3f})")
print(f"95% Confidence Interval (RU): ({lower_ru:.3f}, {upper_ru:.3f})")

In [ ]:

#%%
#phenol_hh_prop = [[0.421144412,2.463593027,1.0546875,1.843867727,2.535460804,1.201173914,3.161764706,0,0,26]]
#phenol_eq_prop = [[2.463593027,2.883230676,2.58236337,2.535460804,0.623015873,0.821520201,0.082861171,3.161764706,1.625732244,0]]
phenol_cc_prop = [[1.78503652,3.35034302,

    ,2.58236337,1.0546875,2.535460804,0.623015873,0.082861171,1.042309493,3.161764706,0,1]]
#IPA_ru_prop = [[2.463593027,2.58236337,1.0546875,1.744142101,2.535460804,1.042309493,1.968534508,1.459430895,0,0]]

# IPA_hh_prop = [[0.421144412,2.463593027,1.0546875,1.843867727,2.535460804,1.201173914,3.161764706,0,0,26]]
# IPA_eq_prop = [[2.463593027,2.883230676,2.58236337,2.535460804,0.623015873,0.821520201,0.082861171,3.161764706,1.625732244,0]]
# IPA_cc_prop = [[2.463593027,2.58236337,1.0546875,2.535460804,0.623015873,0.082861171,1.042309493,3.161764706,0,1]]
# IPA_ru_prop = [[2.463593027,2.58236337,1.0546875,1.744142101,2.535460804,1.042309493,1.968534508,1.459430895,0,0]]

#NMP_hh_prop = [[0.509800001,1.127118601,0.7421875,1.200711595,3.67479588,1.397527416,2.56127451,0,0,40]]
#NMP_eq_prop = [[1.127118601,3.523240577,3.14977878,3.67479588,1.275396825,0.93302656,1.441256716,2.56127451,1.602682581,0]]
#NMP_cc_prop = [[1.127118601,3.523240577,0.7421875,3.67479588,1.275396825,0.93302656,1.730906343,2.56127451,0,0]]
#NMP_ru_prop = [[1.127118601,3.523240577,0.7421875,1.629818594,3.67479588,1.730906343,1.728744575,2.105024424,0,0]]

#HEMA_hh_prop = [[0.364774879,1.476967839,1.1328125,2.208036835,3.43572229,0,3.161764706,0,0,8]]
#HEMA_eq_prop = [[1.476967839,3.690742311,1.80428559,3.43572229,1.305555556,0,0,3.161764706,0.772305235,0]]
#HEMA_cc_prop = [[1.476967839,3.690742311,1.1328125,3.43572229,1.305555556,0,0.300122101,3.161764706,0,1]]
#HEMA_ru_prop = [[1.476967839,3.690742311,1.1328125,1.328396636,3.43572229,0.300122101,2.05754761,-0.251441668,0,0]]

#TFA_hh_prop = [[0.214700151,1.670769215,1.2890625,1.582251988,2.444740915,-0.032243538,3.125,0,0,8]]
#TFA_eq_prop = [[1.670769215,2.503661451,-0.40947341,2.444740915,0.551587302,0,0,3.125,0.967712782,1]]
#TFA_cc_prop = [[1.670769215,2.503661451,1.2890625,2.444740915,0.551587302,0,0.33534835,3.125,0,0]]
#TFA_ru_prop = [[1.670769215,2.503661451,1.2890625,2.227891156,2.444740915,0.33534835,1.343581594,0.279812401,0,0]]
           
#HCL_hh_prop = [[0.181338286,1.160832565,1.25,0.730431143,1.555579273,0.308535733,3.566176471,0,0,8]]
#HCL_eq_prop = [[1.160832565,1.293070223,3.66396287,1.555579273,0.063492063,0.020715293,0,3.566176471,0.287937483,1]]
#HCL_cc_prop = [[1.160832565,1.293070223,1.25,1.555579273,0.063492063,0.020715293,0.617249087,3.566176471,0,0]]
#HCL_ru_prop = [[1.160832565,1.293070223,1.25,3.394746788,1.555579273,0.617249087,5,5,0,0]]

In [ ]:
#%%

IPA_hh = model_hh.predict(IPA_hh_prop, verbose=0)
IPA_eq = model_eq.predict(IPA_eq_prop, verbose=0)
IPA_cc = model_cc.predict(IPA_cc_prop, verbose=0)
IPA_ru = model_ru.predict(IPA_ru_prop, verbose=0)

# #NMP_hh = model_hh.predict(NMP_hh_prop, verbose=0)
# #NMP_eq = model_eq.predict(NMP_eq_prop, verbose=0)
# #NMP_cc = model_cc.predict(NMP_cc_prop, verbose=0)
# #NMP_ru = model_ru.predict(NMP_ru_prop, verbose=0)

# HEMA_hh = model_hh.predict(HEMA_hh_prop, verbose=0)
# HEMA_eq = model_eq.predict(HEMA_eq_prop, verbose=0)
# HEMA_cc = model_cc.predict(HEMA_cc_prop, verbose=0)
# HEMA_ru = model_ru.predict(HEMA_ru_prop, verbose=0)

# TFA_hh = model_hh.predict(TFA_hh_prop, verbose=0)
# TFA_eq = model_eq.predict(TFA_eq_prop, verbose=0)
# TFA_cc = model_cc.predict(TFA_cc_prop, verbose=0)
# TFA_ru = model_ru.predict(TFA_ru_prop, verbose=0)

# HCL_hh = model_hh.predict(HCL_hh_prop, verbose=0)
# HCL_eq = model_eq.predict(HCL_eq_prop, verbose=0)
# HCL_cc = model_cc.predict(HCL_cc_prop, verbose=0)
# HCL_ru = model_ru.predict(HCL_ru_prop, verbose=0)

print(IPA_hh.item(), IPA_eq.item(), IPA_cc.item(), IPA_ru.item())
# print(HEMA_hh.item(), HEMA_eq.item(), HEMA_cc.item(), HEMA_ru.item())
# print(TFA_hh.item(), TFA_eq.item(), TFA_cc.item(), TFA_ru.item())
# print(HCL_hh.item(), HCL_eq.item(), HCL_cc.item(), HCL_ru.item())

In [ ]:

ef sensitivity_analysis(model, input_data, feature_index, perturb_range):
    """
    Performs sensitivity analysis on a specific feature.
    
    Args:
    - model: The trained ANN model
    - input_data: The baseline input data (e.g., X_test)
    - feature_index: Index of the feature to perturb
    - perturb_range: Range of values to perturb the feature
    
    Returns:
    - A tuple of perturbed values and corresponding model outputs
    """
    
    # Copy the input data to avoid modifying the original
    perturbed_data = np.copy(input_data)
    
    # Store the model outputs for each perturbation
    outputs = []
    
    for value in perturb_range:
        perturbed_data[:, feature_index] = value
        model_output = model_hh.predict(perturbed_data)
        
        # Here, we'll assume the model is doing regression and has a single output
        # For classification, you might want to store probabilities for a specific class
        outputs.append(model_output[:, 0])
    
    return perturb_range, outputs

# Choose a feature index to analyze (e.g., the 0-th feature)
feature_idx = 3

# Define the range of values to perturb the feature
# You can adjust this range based on the specific feature's domain
perturb_values = np.linspace(start=Xtest_HH.iloc[:, feature_idx].min(), 
                             stop=Xtest_HH.iloc[:, feature_idx].max(), 
                             num=100)

values, model_outputs = sensitivity_analysis(model_hh, Xtest_HH, feature_idx, perturb_values)

In [ ]:
#%%

# Plot the results
plt.figure(figsize=(8,6), dpi=600) 
plt.plot(values, model_outputs)
plt.xlabel('Feature Value')
plt.ylabel('Model Output')
plt.title(f'Sensitivity Analysis for Feature {sel_data_HH.columns[3]}')
plt.grid(True)
plt.show()
